# Google Colab Pro Training Setup for Moisture Detection Model (Simplified Structure)

## Overview

This notebook provides step-by-step instructions to train a custom image classification model using Google Colab Pro with your 23,000+ labeled images from Google Drive with simplified structure (after restructuring), then convert it for use with your existing TensorFlow.js application.

## Prerequisites

- Google Colab Pro subscription ($10/month)
- Google Drive with organized image dataset in simplified structure:
  ```
  Predicto_GPT_Taining_images/
  ├── 0/
  │   ├── image1_day.jpg
  │   ├── image2_night.jpg
  │   └── ...
  ├── 25/
  │   ├── photo_1_day.png
  │   ├── photo_2_night.png
  │   └── ...
  ├── ... (50, 75, 100, 130, 175, 200, 250, 300, 350, 400, 450)
  └── Invalid/
      ├── invalid_image1.jpg
      └── ...
  ```
- Basic understanding of Python and machine learning concepts

## Expected Results

- **Training Time**: 2-4 hours on Colab Pro GPU
- **Model Accuracy**: Typically 85-95% with proper data
- **Output**: TensorFlow.js compatible model files
- **Model Size**: 5-15MB (with quantization)

## Step 1: Setup Google Colab Pro Environment

### 1.1 Subscribe and Access

1. Go to [Google Colab](https://colab.research.google.com)
2. Subscribe to Colab Pro ($10/month) for GPU access and longer runtimes
3. Create a new notebook: **File → New notebook**

### 1.2 Configure Runtime

1. **Runtime → Change runtime type**
2. Set **Hardware accelerator** to **GPU**
3. Set **Runtime shape** to **High-RAM** (if available)
4. Click **Save**

### 1.3 Verify GPU Access

In [ ]:
# Check GPU availability
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

# Get GPU details
if tf.config.list_physical_devices('GPU'):
    gpu = tf.config.experimental.get_device_details(tf.config.list_physical_devices('GPU')[0])
    print("GPU details:", gpu)
else:
    print("⚠️  No GPU detected - check runtime settings")

## Step 2: Mount Google Drive and Install Dependencies

### 2.1 Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Verify mount
print("✅ Google Drive mounted successfully")
print("Available folders:", os.listdir('/content/drive/MyDrive')[:10])

### 2.1.5 Understanding Dependency Conflicts

**📋 Common Issue**: Google Colab sometimes has package version conflicts. If you encounter NumPy/scikit-learn compatibility errors, here are the solutions in order of preference:

1. **Recommended**: Use the installation approach below with specific version constraints
2. **Fallback**: Skip scikit-learn installation - the notebook includes built-in alternatives
3. **Last Resort**: Use only essential packages and minimal functionality

### 2.2 Install Required Packages with Dependency Resolution

In [ ]:
# Check current versions and conflicts
print("🔍 Checking current package versions...")
!pip list | grep -E "(numpy|tensorflow|scikit-learn|opencv)"

print("\n📦 Installing packages with careful dependency management...")

# Install core packages first with compatible versions
!pip install -q tensorflowjs matplotlib pillow pandas

# Install compatible versions that work together
!pip install -q "numpy>=1.26.0,<2.0"  # Compatible with TensorFlow 2.19
!pip install -q seaborn

# Use the scikit-learn that works with the installed numpy
!pip install -q --no-deps scikit-learn  # Install without dependencies first
!pip install -q "scikit-learn>=1.3.0,<1.4.0" --force-reinstall

print("✅ Package installation complete!")
print("🔄 Restarting runtime for clean imports...")

# Restart runtime to ensure clean imports
import os
os.kill(os.getpid(), 9)

**⚠️ IMPORTANT: After running the above cell, the runtime will restart. Run the cell below in a NEW cell:**

**Alternative approach if dependency conflicts persist:**

In [ ]:
# If you're still getting conflicts, use this minimal installation approach
print("🚨 If dependency conflicts persist, run this alternative installation:")
print("!pip install -q tensorflowjs matplotlib pillow pandas seaborn")
print("# Skip scikit-learn installation and use fallback functions")
print("# The notebook includes built-in alternatives for train_test_split and class_weight")

In [ ]:
# Import essential libraries (run this after runtime restart)
print("🔍 Checking TensorFlow and NumPy versions...")
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

# Verify compatibility
if hasattr(tf, 'config') and tf.config.list_physical_devices('GPU'):
    print("✅ GPU detected and accessible")
else:
    print("⚠️ No GPU detected - check runtime settings")

# Handle seaborn import with fallback
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
    print("✅ Seaborn imported successfully")
except ImportError as e:
    print(f"⚠️ Seaborn import failed: {e}")
    print("Will use matplotlib for all visualizations")
    SEABORN_AVAILABLE = False

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import tensorflowjs as tfjs
from datetime import datetime
import json
from collections import Counter
# Try importing sklearn - use fallback if it fails
try:
    from sklearn.model_selection import train_test_split
    from sklearn.utils.class_weight import compute_class_weight
    SKLEARN_AVAILABLE = True
    print("✅ Scikit-learn imported successfully")
except ImportError as e:
    print(f"⚠️  Scikit-learn import failed: {e}")
    print("Using built-in alternatives for train_test_split and class_weight")
    SKLEARN_AVAILABLE = False
import shutil
import glob
from pathlib import Path

# Fallback functions if sklearn is not available
if not SKLEARN_AVAILABLE:
    def train_test_split(*arrays, test_size=None, train_size=None, stratify=None, random_state=None):
        """Simple fallback for train_test_split with stratification"""
        if random_state:
            np.random.seed(random_state)
        
        # Get the main array (first one)
        main_array = arrays[0]
        n_samples = len(main_array)
        
        # Calculate test size
        if test_size is None:
            test_size = 0.25
        if test_size < 1:
            test_size = int(test_size * n_samples)
        
        # Create indices
        indices = np.arange(n_samples)
        
        if stratify is not None:
            # Stratified splitting
            unique_labels = np.unique(stratify)
            train_indices = []
            test_indices = []
            
            for label in unique_labels:
                label_indices = indices[stratify == label]
                np.random.shuffle(label_indices)
                
                label_test_size = int(len(label_indices) * (test_size / n_samples))
                
                test_indices.extend(label_indices[:label_test_size])
                train_indices.extend(label_indices[label_test_size:])
        else:
            # Simple random splitting
            np.random.shuffle(indices)
            test_indices = indices[:test_size]
            train_indices = indices[test_size:]
        
        # Split all arrays
        result = []
        for array in arrays:
            array = np.array(array)
            result.extend([array[train_indices], array[test_indices]])
        
        return result
    
    def compute_class_weight(class_weight, classes, y):
        """Simple fallback for compute_class_weight"""
        if class_weight == 'balanced':
            # Calculate balanced weights
            unique_classes, counts = np.unique(y, return_counts=True)
            total_samples = len(y)
            n_classes = len(unique_classes)
            
            weights = total_samples / (n_classes * counts)
            return weights
        else:
            return np.ones(len(classes))

print("✅ All packages imported successfully")

## Step 3: Data Loading for Simplified Structure

### 3.1 Set Data Paths and Configuration

In [ ]:
# Configure paths (UPDATE THESE TO MATCH YOUR GOOGLE DRIVE STRUCTURE)
DRIVE_DATA_PATH = '/content/drive/MyDrive/Predicto_GPT_Taining_images'  # ⚠️ UPDATE THIS PATH
MODEL_NAME = 'iron-custom-v2.0'  # Change as needed
WORK_DIR = '/content/moisture_detection_training'

# Data split configuration
TRAIN_SPLIT = 0.7   # 70% for training
VAL_SPLIT = 0.2     # 20% for validation
TEST_SPLIT = 0.1    # 10% for testing

# Data sampling configuration
MAX_IMAGES_PER_CLASS = 200  # ⚠️ CONFIGURABLE: Limit images per class (set to None for all images)

# Define your 14 classes
CLASS_NAMES = ['0', '25', '50', '75', '100', '130', '175', '200', '250', '300', '350', '400', '450', 'Invalid']

# Create working directory
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Working directory: {WORK_DIR}")
print(f"Data path: {DRIVE_DATA_PATH}")
print(f"Classes: {CLASS_NAMES}")
print(f"Data splits: Train={TRAIN_SPLIT}, Val={VAL_SPLIT}, Test={TEST_SPLIT}")
if MAX_IMAGES_PER_CLASS:
    print(f"Max images per class: {MAX_IMAGES_PER_CLASS}")
else:
    print("Using all available images per class")

### 3.2 Simplified Data Collection Function

In [ ]:
def collect_image_paths_and_labels(data_path, class_names, max_images_per_class=None):
    """
    Collect image paths and their corresponding labels from simplified structure.
    All images are directly in class folders with _day/_night suffixes.
    
    Args:
        data_path: Path to the root data directory
        class_names: List of class names
        max_images_per_class: Maximum number of images to collect per class (None for all)
    """
    image_paths = []
    labels = []
    class_counts = {}
    
    print("🔍 Scanning simplified dataset structure...")
    if max_images_per_class:
        print(f"🎯 Limiting to {max_images_per_class} images per class")
    
    for class_idx, class_name in enumerate(class_names):
        class_path = os.path.join(data_path, class_name)
        
        if not os.path.exists(class_path):
            print(f"⚠️  Warning: Class folder '{class_name}' not found at {class_path}")
            class_counts[class_name] = 0
            continue
            
        class_image_count = 0
        class_image_paths = []  # Collect all paths for this class first
        
        # Get all image files directly from class folder
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']:
            pattern = os.path.join(class_path, ext)
            found_files = glob.glob(pattern)
            class_image_paths.extend(found_files)
        
        # Shuffle and limit the images for this class if max_images_per_class is set
        if class_image_paths:
            # Shuffle to get random selection
            np.random.shuffle(class_image_paths)
            
            # Limit to max_images_per_class if specified
            if max_images_per_class:
                class_image_paths = class_image_paths[:max_images_per_class]
            
            # Add to final lists
            for img_path in class_image_paths:
                image_paths.append(img_path)
                labels.append(class_idx)
                class_image_count += 1
        
        class_counts[class_name] = class_image_count
        total_available = len(glob.glob(os.path.join(class_path, '*.*'))) if os.path.exists(class_path) else 0
        
        if max_images_per_class and total_available > max_images_per_class:
            print(f"  {class_name:>10}: {class_image_count:>6,} images (limited from {total_available:,} available)")
        else:
            print(f"  {class_name:>10}: {class_image_count:>6,} images")
        
        # Show day/night distribution for this class
        day_count = len([p for p in class_image_paths if '_day.' in os.path.basename(p)])
        night_count = len([p for p in class_image_paths if '_night.' in os.path.basename(p)])
        other_count = class_image_count - day_count - night_count
        
        if day_count > 0 or night_count > 0:
            print(f"    └─ Day: {day_count}, Night: {night_count}" + (f", Other: {other_count}" if other_count > 0 else ""))
    
    return image_paths, labels, class_counts

# Set random seed for reproducible sampling
np.random.seed(42)

# Collect all image paths and labels
print("📊 Collecting image paths from simplified structure...")
all_image_paths, all_labels, class_counts = collect_image_paths_and_labels(
    DRIVE_DATA_PATH, CLASS_NAMES, MAX_IMAGES_PER_CLASS
)

total_images = len(all_image_paths)
print(f"\n✅ Dataset collection complete!")
print(f"Total images: {total_images:,}")
print(f"Total classes: {len(CLASS_NAMES)}")
if MAX_IMAGES_PER_CLASS:
    expected_max = len(CLASS_NAMES) * MAX_IMAGES_PER_CLASS
    print(f"Expected maximum (if all classes full): {expected_max:,} images")

# Verify we have data for all classes
empty_classes = [name for name, count in class_counts.items() if count == 0]
if empty_classes:
    print(f"⚠️  Empty classes found: {empty_classes}")
else:
    print("✅ All classes have images")

### 3.3 Dataset Analysis and Visualization

In [ ]:
def analyze_dataset(image_paths, labels, class_names, class_counts):
    """Comprehensive dataset analysis"""
    
    # Convert to numpy arrays for easier manipulation
    labels_array = np.array(labels)
    
    print(f"\n📊 Detailed Dataset Analysis:")
    print("=" * 60)
    print(f"Total images: {len(image_paths):,}")
    print(f"Total classes: {len(class_names)}")
    
    # Class distribution
    print(f"\n📋 Class distribution:")
    for i, class_name in enumerate(class_names):
        count = class_counts[class_name]
        percentage = (count / len(image_paths)) * 100 if len(image_paths) > 0 else 0
        print(f"  {class_name:>10}: {count:>6,} images ({percentage:>5.1f}%)")
    
    # Check for class imbalance
    counts = list(class_counts.values())
    non_zero_counts = [c for c in counts if c > 0]
    
    if non_zero_counts:
        min_count = min(non_zero_counts)
        max_count = max(non_zero_counts)
        imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
        
        print(f"\n⚖️  Class balance analysis:")
        print(f"Min class size: {min_count:,}")
        print(f"Max class size: {max_count:,}")
        print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")
        
        if imbalance_ratio > 5:
            print("⚠️  Significant class imbalance detected!")
            print("Consider using class weights during training.")
    
    # Visualize class distribution
    plt.figure(figsize=(15, 6))
    classes = list(class_counts.keys())
    counts = list(class_counts.values())
    
    plt.bar(classes, counts)
    plt.title('Class Distribution in Dataset')
    plt.xlabel('Classes')
    plt.ylabel('Number of Images')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    return labels_array

# Analyze the collected dataset
labels_array = analyze_dataset(all_image_paths, all_labels, CLASS_NAMES, class_counts)

### 3.4 Stratified Train/Validation/Test Split

In [ ]:
def create_stratified_splits(image_paths, labels, class_names, train_split=0.7, val_split=0.2, test_split=0.1):
    """
    Create stratified train/validation/test splits ensuring each class is represented
    in all splits according to its distribution in the original dataset.
    """
    
    print("🔄 Creating stratified train/validation/test splits...")
    
    # Verify splits sum to 1
    total_split = train_split + val_split + test_split
    if abs(total_split - 1.0) > 0.001:
        raise ValueError(f"Splits must sum to 1.0, got {total_split}")
    
    # Convert to numpy arrays
    image_paths = np.array(image_paths)
    labels = np.array(labels)
    
    # First split: separate train from temp (val + test)
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels,
        test_size=(val_split + test_split),
        stratify=labels,
        random_state=42
    )
    
    # Second split: separate validation from test
    # Calculate relative sizes for val and test from the temp set
    val_relative_size = val_split / (val_split + test_split)
    
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels,
        test_size=(1 - val_relative_size),
        stratify=temp_labels,
        random_state=42
    )
    
    # Verify splits
    total_samples = len(image_paths)
    
    print(f"\n✅ Split creation complete:")
    print(f"Training samples:   {len(train_paths):>6,} ({len(train_paths)/total_samples*100:.1f}%)")
    print(f"Validation samples: {len(val_paths):>6,} ({len(val_paths)/total_samples*100:.1f}%)")
    print(f"Test samples:       {len(test_paths):>6,} ({len(test_paths)/total_samples*100:.1f}%)")
    print(f"Total samples:      {total_samples:>6,}")
    
    # Verify stratification worked
    print(f"\n📊 Per-class distribution verification:")
    for i, class_name in enumerate(class_names):
        train_count = np.sum(train_labels == i)
        val_count = np.sum(val_labels == i)
        test_count = np.sum(test_labels == i)
        total_count = train_count + val_count + test_count
        
        if total_count > 0:
            print(f"  {class_name:>10}: Train={train_count:>4} ({train_count/total_count*100:>4.1f}%) "
                  f"Val={val_count:>4} ({val_count/total_count*100:>4.1f}%) "
                  f"Test={test_count:>4} ({test_count/total_count*100:>4.1f}%)")
    
    return (train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels)

# Create the splits
(train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels) = create_stratified_splits(
    all_image_paths, all_labels, CLASS_NAMES, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT
)

## Step 4: Custom Data Pipeline with TensorFlow Dataset

### 4.1 Image Loading and Preprocessing Functions

In [ ]:
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    """Load and preprocess a single image"""
    # Load image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.cast(image, tf.float32)
    
    # Ensure image has correct shape
    image.set_shape([None, None, 3])
    
    # Resize to target size
    image = tf.image.resize(image, target_size)
    
    # Normalize to [0, 1]
    image = image / 255.0
    
    return image

def augment_image(image, label):
    """Apply data augmentation to training images"""
    # Random rotation
    image = tf.image.rot90(image, tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))
    
    # Random flip
    image = tf.image.random_flip_left_right(image)
    
    # Random brightness
    image = tf.image.random_brightness(image, max_delta=0.2)
    
    # Random contrast
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    
    # Random zoom (crop and resize)
    shape = tf.shape(image)
    crop_size = tf.random.uniform([], 0.8, 1.0) * tf.cast(tf.minimum(shape[0], shape[1]), tf.float32)
    crop_size = tf.cast(crop_size, tf.int32)
    
    image = tf.image.random_crop(image, [crop_size, crop_size, 3])
    image = tf.image.resize(image, [224, 224])
    
    # Ensure values are still in [0, 1] range
    image = tf.clip_by_value(image, 0.0, 1.0)
    
    return image, label

def create_dataset(image_paths, labels, class_names, batch_size=32, shuffle=True, augment=False):
    """Create a TensorFlow dataset from image paths and labels"""
    
    print(f"🔄 Creating dataset with {len(image_paths):,} images...")
    
    # Convert to tensors
    path_tensor = tf.constant(image_paths)
    label_tensor = tf.constant(labels)
    
    # Create dataset from tensor slices
    dataset = tf.data.Dataset.from_tensor_slices((path_tensor, label_tensor))
    
    # Shuffle if requested
    if shuffle:
        print("🔀 Shuffling dataset...")
        dataset = dataset.shuffle(buffer_size=len(image_paths))
    
    # Map image loading function
    print("📸 Loading and preprocessing images...")
    dataset = dataset.map(
        lambda path, label: (load_and_preprocess_image(path), tf.one_hot(label, len(class_names))),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    # Apply augmentation if requested
    if augment:
        print("🎨 Applying data augmentation...")
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Filter out any problematic samples
    dataset = dataset.filter(lambda image, label: tf.reduce_all(tf.math.is_finite(image)))
    
    # Batch and prefetch
    print(f"📦 Batching (batch_size={batch_size}) and prefetching...")
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    print("✅ Dataset creation completed!")
    return dataset

print("🔄 Creating TensorFlow datasets...")

# Create datasets
BATCH_SIZE = 32

# Test with a small sample first to verify everything works
print("🧪 Testing dataset creation with 5 samples...")
test_paths_small = train_paths[:5]
test_labels_small = train_labels[:5]
test_dataset_small = create_dataset(
    test_paths_small, test_labels_small, CLASS_NAMES,
    batch_size=2, shuffle=False, augment=False
)
print("✅ Small test successful!")

# Now create full datasets
print("\n📊 Creating full datasets...")
import time

start_time = time.time()
train_dataset = create_dataset(
    train_paths, train_labels, CLASS_NAMES, 
    batch_size=BATCH_SIZE, shuffle=True, augment=True
)
train_time = time.time() - start_time
print(f"⏱️  Training dataset created in {train_time:.1f} seconds")

start_time = time.time()
val_dataset = create_dataset(
    val_paths, val_labels, CLASS_NAMES,
    batch_size=BATCH_SIZE, shuffle=False, augment=False
)
val_time = time.time() - start_time
print(f"⏱️  Validation dataset created in {val_time:.1f} seconds")

start_time = time.time()
test_dataset = create_dataset(
    test_paths, test_labels, CLASS_NAMES,
    batch_size=BATCH_SIZE, shuffle=False, augment=False
)
test_time = time.time() - start_time
print(f"⏱️  Test dataset created in {test_time:.1f} seconds")

print("✅ All datasets created successfully!")

# Get dataset sizes (this will actually iterate through datasets)
print("📏 Calculating dataset sizes...")
train_batches = tf.data.experimental.cardinality(train_dataset).numpy()
val_batches = tf.data.experimental.cardinality(val_dataset).numpy()
test_batches = tf.data.experimental.cardinality(test_dataset).numpy()

print(f"Training batches: {train_batches}")
print(f"Validation batches: {val_batches}")
print(f"Test batches: {test_batches}")

### 4.2 Dataset Visualization

In [ ]:
def visualize_dataset_samples(dataset, class_names, num_samples=12):
    """Visualize sample images from the dataset"""
    
    # Get a batch from the dataset
    sample_batch = next(iter(dataset))
    images, labels = sample_batch
    
    # Convert one-hot labels back to class indices
    label_indices = tf.argmax(labels, axis=1)
    
    # Plot samples
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()
    
    for i in range(min(num_samples, len(images))):
        img = images[i].numpy()
        label_idx = label_indices[i].numpy()
        class_name = class_names[label_idx]
        
        axes[i].imshow(img)
        axes[i].set_title(f'Class: {class_name}', fontsize=12)
        axes[i].axis('off')
    
    # Hide unused subplots
    for i in range(min(num_samples, len(images)), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Dataset Samples (with Augmentation for Training)', fontsize=16)
    plt.tight_layout()
    plt.show()

print("🖼️  Visualizing training dataset samples:")
visualize_dataset_samples(train_dataset, CLASS_NAMES)

print("🖼️  Visualizing validation dataset samples:")
visualize_dataset_samples(val_dataset, CLASS_NAMES)

### 4.3 Calculate Class Weights for Imbalanced Data

In [ ]:
def calculate_class_weights(labels, class_names):
    """Calculate class weights to handle imbalanced dataset"""
    
    # Calculate class weights
    unique_labels = np.unique(labels)
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=unique_labels,
        y=labels
    )
    
    # Create class weight dictionary
    class_weight_dict = {i: weight for i, weight in zip(unique_labels, class_weights)}
    
    print("⚖️  Class weights for handling imbalance:")
    for i, class_name in enumerate(class_names):
        if i in class_weight_dict:
            print(f"  {class_name:>10}: {class_weight_dict[i]:.3f}")
    
    return class_weight_dict

# Calculate class weights
class_weights = calculate_class_weights(train_labels, CLASS_NAMES)

## Step 5: Build Advanced Model Architecture

### 5.1 Create Model

In [ ]:
def create_model(num_classes, input_shape=(224, 224, 3)):
    """Create MobileNetV2-based transfer learning model"""
    
    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        alpha=1.0  # Width multiplier
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Add custom classification head with explicit input specification for TensorFlow.js compatibility
    inputs = tf.keras.Input(shape=input_shape, batch_size=None, name='input_layer')
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = Dropout(0.2, name='dropout_1')(x)
    x = Dense(128, activation='relu', name='dense_intermediate')(x)
    x = Dropout(0.5, name='dropout_2')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='iron_custom_model')
    return model, base_model

# Create model
print("🏗️  Building model architecture...")
model, base_model = create_model(len(CLASS_NAMES))

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

print("✅ Model created and compiled")
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.reduce_prod(var.shape) for var in model.trainable_variables]):,}")

# Display model summary
model.summary()

## Step 6: Advanced Training with Callbacks

### 6.1 Setup Training Callbacks

In [ ]:
# Create callbacks for training optimization
callbacks = [
    # Early stopping to prevent overfitting
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate when plateau
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Save best model
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),
    
    # CSV logger for training history
    tf.keras.callbacks.CSVLogger('training_log.csv'),
]

print("✅ Training callbacks configured")

### 6.2 Phase 1: Train with Frozen Base

In [ ]:
print("\n🚀 Phase 1: Training with frozen base model...")
print("=" * 60)

# Record start time
import time
phase1_start = time.time()

# Train with frozen base model
history1 = model.fit(
    train_dataset,
    epochs=25,
    validation_data=val_dataset,
    callbacks=callbacks,
    class_weight=class_weights,  # Use class weights for imbalanced data
    verbose=1
)

phase1_duration = time.time() - phase1_start
print(f"\n⏱️  Phase 1 completed in {phase1_duration/60:.1f} minutes")

# Plot training progress
def plot_training_history(history, title="Training History"):
    """Plot training and validation metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)
    
    # Loss
    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_training_history(history1, "Phase 1: Frozen Base Model")

### 6.3 Phase 2: Fine-tuning with Unfrozen Base

In [ ]:
print("\n🔥 Phase 2: Fine-tuning with unfrozen base model...")
print("=" * 60)

# Unfreeze base model for fine-tuning
base_model.trainable = True

# Compile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),  # Lower LR
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

print(f"Trainable parameters after unfreezing: {sum([tf.reduce_prod(var.shape) for var in model.trainable_variables]):,}")

# Record start time
phase2_start = time.time()

# Fine-tune the model
history2 = model.fit(
    train_dataset,
    epochs=40,
    validation_data=val_dataset,
    callbacks=callbacks,
    class_weight=class_weights,  # Continue using class weights
    verbose=1
)

phase2_duration = time.time() - phase2_start
total_training_time = phase1_duration + phase2_duration

print(f"\n⏱️  Phase 2 completed in {phase2_duration/60:.1f} minutes")
print(f"🎯 Total training time: {total_training_time/60:.1f} minutes")

# Plot fine-tuning progress
plot_training_history(history2, "Phase 2: Fine-tuning")

## Step 7: Model Evaluation and Testing

### 7.1 Load Best Model and Evaluate

In [ ]:
# Load the best saved model
print("📈 Loading best model for evaluation...")
best_model = tf.keras.models.load_model('best_model.h5')

# Evaluate on validation set
print("🧪 Evaluating model performance on validation set...")
val_results = best_model.evaluate(val_dataset, verbose=1)
val_loss, val_accuracy, val_top3_acc = val_results

# Evaluate on test set
print("🧪 Evaluating model performance on test set...")
test_results = best_model.evaluate(test_dataset, verbose=1)
test_loss, test_accuracy, test_top3_acc = test_results

print(f"\n🎯 Final Model Performance:")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
print(f"Validation Top-3 Accuracy: {val_top3_acc:.4f} ({val_top3_acc*100:.2f}%)")
print()
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test Top-3 Accuracy: {test_top3_acc:.4f} ({test_top3_acc*100:.2f}%)")

### 7.2 Detailed Classification Report

In [ ]:
# Try importing sklearn metrics - use fallback if not available
try:
    from sklearn.metrics import classification_report, confusion_matrix
except ImportError:
    print("⚠️  sklearn.metrics not available - using simplified alternatives")
    
    def classification_report(y_true, y_pred, target_names=None, digits=4):
        """Simple fallback for classification_report"""
        unique_labels = np.unique(y_true)
        report_lines = []
        
        report_lines.append("Classification Report")
        report_lines.append("=" * 50)
        report_lines.append(f"{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
        report_lines.append("-" * 60)
        
        total_correct = 0
        total_samples = 0
        
        for i, label in enumerate(unique_labels):
            # Calculate metrics for this class
            true_positives = np.sum((y_true == label) & (y_pred == label))
            false_positives = np.sum((y_true != label) & (y_pred == label))
            false_negatives = np.sum((y_true == label) & (y_pred != label))
            
            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            support = np.sum(y_true == label)
            
            class_name = target_names[i] if target_names else str(label)
            report_lines.append(f"{class_name:<15} {precision:<10.{digits}f} {recall:<10.{digits}f} {f1:<10.{digits}f} {support:<10}")
            
            total_correct += true_positives
            total_samples += support
        
        accuracy = total_correct / total_samples if total_samples > 0 else 0
        report_lines.append("-" * 60)
        report_lines.append(f"{'accuracy':<15} {'':<10} {'':<10} {accuracy:<10.{digits}f} {total_samples:<10}")
        
        return "\n".join(report_lines)
    
    def confusion_matrix(y_true, y_pred):
        """Simple fallback for confusion_matrix"""
        unique_labels = np.unique(np.concatenate([y_true, y_pred]))
        n_labels = len(unique_labels)
        cm = np.zeros((n_labels, n_labels), dtype=int)
        
        for i, true_label in enumerate(unique_labels):
            for j, pred_label in enumerate(unique_labels):
                cm[i, j] = np.sum((y_true == true_label) & (y_pred == pred_label))
        
        return cm

def get_predictions_and_labels(model, dataset):
    """Get model predictions and true labels from dataset"""
    predictions = []
    true_labels = []
    
    for batch_images, batch_labels in dataset:
        batch_predictions = model.predict(batch_images, verbose=0)
        predictions.extend(batch_predictions)
        true_labels.extend(tf.argmax(batch_labels, axis=1).numpy())
    
    return np.array(predictions), np.array(true_labels)

# Get predictions for test set
print("🔍 Generating predictions for detailed analysis...")
test_predictions, test_true_labels = get_predictions_and_labels(best_model, test_dataset)
test_predicted_classes = np.argmax(test_predictions, axis=1)

# Generate classification report
print("\n📋 Detailed Classification Report (Test Set):")
print("=" * 80)
report = classification_report(
    test_true_labels,
    test_predicted_classes,
    target_names=CLASS_NAMES,
    digits=4
)
print(report)

# Save report to file
with open(f'/content/drive/MyDrive/{MODEL_NAME}_classification_report.txt', 'w') as f:
    f.write(f"Classification Report for {MODEL_NAME}\n")
    f.write("=" * 80 + "\n")
    f.write(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")
    f.write(f"Test Top-3 Accuracy: {test_top3_acc:.4f} ({test_top3_acc*100:.2f}%)\n\n")
    f.write(report)

print(f"✅ Classification report saved to Google Drive")

### 7.3 Confusion Matrix Visualization

In [ ]:
# Create and visualize confusion matrix
cm = confusion_matrix(test_true_labels, test_predicted_classes)

# Calculate accuracy per class
class_accuracies = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(14, 12))

# Create heatmap - use seaborn if available, matplotlib otherwise
if SEABORN_AVAILABLE:
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cmap='Blues',
        cbar_kws={'label': 'Count'}
    )
else:
    # Fallback to matplotlib
    im = plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, label='Count')
    
    # Add text annotations
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    
    # Set labels
    tick_marks = np.arange(len(CLASS_NAMES))
    plt.xticks(tick_marks, CLASS_NAMES, rotation=45)
    plt.yticks(tick_marks, CLASS_NAMES, rotation=0)

plt.title(f'Confusion Matrix - {MODEL_NAME}\nTest Accuracy: {test_accuracy:.2%}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()

# Save confusion matrix
plt.savefig(f'/content/drive/MyDrive/{MODEL_NAME}_confusion_matrix.png',
            dpi=300, bbox_inches='tight')
plt.show()

# Print per-class accuracies
print("\n🎯 Per-class accuracies (Test Set):")
for i, (class_name, accuracy) in enumerate(zip(CLASS_NAMES, class_accuracies)):
    if not np.isnan(accuracy):
        print(f"{class_name:>10}: {accuracy:.2%} ({cm[i,i]:4d}/{cm[i].sum():4d})")
    else:
        print(f"{class_name:>10}: No samples in test set")

## Step 8: Convert to TensorFlow.js Format

### 8.1 Convert and Save Model

In [ ]:
# Convert model to TensorFlow.js format
print("🔄 Converting model to TensorFlow.js format...")

# Create output directory in Google Drive
tfjs_output_path = f'/content/drive/MyDrive/{MODEL_NAME}_tfjs'
os.makedirs(tfjs_output_path, exist_ok=True)

# Convert to TensorFlow.js format with proper configuration
try:
    tfjs.converters.save_keras_model(
        best_model,
        tfjs_output_path,
        quantization_bytes=2,  # Use 16-bit quantization for smaller size
        metadata={'name': MODEL_NAME, 'version': '2.0.0'},
        save_traces=False  # Helps with InputLayer compatibility
    )
    print("✅ TensorFlow.js conversion successful!")
    
    # Verify the model.json file was created properly
    model_json_path = os.path.join(tfjs_output_path, 'model.json')
    if os.path.exists(model_json_path):
        print("✅ model.json file verified")
    else:
        raise Exception("model.json file not found after conversion")
        
except Exception as conversion_error:
    print(f"❌ Initial conversion failed: {conversion_error}")
    print("🔄 Attempting alternative conversion approach...")
    
    # Alternative approach: Create a clean model for conversion
    try:
        # Create new input with explicit specification
        clean_input = tf.keras.Input(
            shape=(224, 224, 3),
            batch_size=None,
            dtype=tf.float32,
            name='input_1'
        )
        
        # Get the model's prediction using the clean input
        clean_output = best_model(clean_input)
        
        # Create a new clean model
        clean_model = tf.keras.Model(
            inputs=clean_input,
            outputs=clean_output,
            name='clean_iron_model'
        )
        
        # Convert the clean model
        tfjs.converters.save_keras_model(
            clean_model,
            tfjs_output_path,
            quantization_bytes=2,
            metadata={'name': MODEL_NAME, 'version': '2.0.0'},
            save_traces=False
        )
        
        print("✅ Alternative conversion approach successful!")
        
    except Exception as alt_error:
        print(f"❌ Alternative conversion also failed: {alt_error}")
        print("⚠️  Manual intervention may be required for model conversion")

print(f"✅ Model converted and saved to: {tfjs_output_path}")

# Check output files
output_files = os.listdir(tfjs_output_path)
print(f"\n📁 Generated files:")
for file in output_files:
    file_path = os.path.join(tfjs_output_path, file)
    file_size = os.path.getsize(file_path)
    print(f"  {file}: {file_size/1024/1024:.2f} MB")

total_size = sum(os.path.getsize(os.path.join(tfjs_output_path, f))
                for f in output_files)
print(f"\n💾 Total model size: {total_size/1024/1024:.2f} MB")

### 8.2 Create Metadata File

In [ ]:
# Generate comprehensive metadata for your application
metadata = {
    "modelInfo": {
        "name": MODEL_NAME,
        "version": "2.0.0",
        "description": f"Custom TensorFlow/Keras model trained on {total_images:,} images with nested Day/Night structure",
        "modelType": "Custom TensorFlow/Keras Image Classification",
        "architecture": "MobileNetV2 + Custom Head",
        "trainingDate": datetime.now().strftime("%Y-%m-%d"),
        "trainingDuration": f"{total_training_time/3600:.1f} hours"
    },
    "performance": {
        "testAccuracy": f"{test_accuracy:.4f}",
        "testAccuracyPercent": f"{test_accuracy*100:.2f}%",
        "testTop3Accuracy": f"{test_top3_acc:.4f}",
        "valAccuracy": f"{val_accuracy:.4f}",
        "valAccuracyPercent": f"{val_accuracy*100:.2f}%",
        "testLoss": f"{test_loss:.4f}",
        "trainingSamples": len(train_paths),
        "validationSamples": len(val_paths),
        "testSamples": len(test_paths),
        "totalSamples": total_images
    },
    "classes": CLASS_NAMES,
    "classIndices": {class_name: i for i, class_name in enumerate(CLASS_NAMES)},
    "modelSpecs": {
        "inputShape": [224, 224, 3],
        "outputShape": [len(CLASS_NAMES)],
        "totalParameters": int(best_model.count_params()),
        "modelSizeMB": round(total_size/1024/1024, 2),
        "quantizationBits": 16
    },
    "trainingConfig": {
        "batchSize": BATCH_SIZE,
        "imageSize": [224, 224],
        "augmentation": True,
        "transferLearning": True,
        "baseModel": "MobileNetV2",
        "optimizer": "Adam",
        "loss": "categorical_crossentropy",
        "classWeights": True,
        "dataSplits": {
            "train": TRAIN_SPLIT,
            "validation": VAL_SPLIT,
            "test": TEST_SPLIT
        }
    },
    "dataStructure": {
        "structure": "Simplified flat structure with _day/_night suffixes",
        "dayNightPreservation": "Filename suffixes (_day, _night)",
        "handledInvalidClass": True,
        "stratifiedSplitting": True,
        "classCount": len(CLASS_NAMES)
    }
}

# Save metadata
metadata_path = os.path.join(tfjs_output_path, 'metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Metadata file created")
print("\n📋 Model metadata preview:")
print(json.dumps(metadata, indent=2)[:1500] + "...")

## Step 9: Generate Configuration and Instructions

### 9.1 Create Model Configuration

In [ ]:
# Generate configuration that matches your existing models.json format
model_config_entry = {
    MODEL_NAME: {
        "name": MODEL_NAME,
        "description": f"Custom TensorFlow/Keras model trained on {metadata['performance']['totalSamples']} images with nested Day/Night structure",
        "version": "2.0.0",
        "trainingDate": metadata['modelInfo']['trainingDate'],
        "modelType": "Custom TensorFlow/Keras Image Classification",
        "classes": CLASS_NAMES,
        "urls": {
            "model": f"https://your-hosting-url.com/{MODEL_NAME}/model.json",  # ⚠️ UPDATE THIS
            "metadata": f"https://your-hosting-url.com/{MODEL_NAME}/metadata.json"  # ⚠️ UPDATE THIS
        },
        "performance": {
            "testAccuracy": metadata['performance']['testAccuracyPercent'],
            "valAccuracy": metadata['performance']['valAccuracyPercent'],
            "trainingSamples": metadata['performance']['trainingSamples'],
            "validationSamples": metadata['performance']['validationSamples'],
            "testSamples": metadata['performance']['testSamples']
        }
    }
}

# Save configuration
config_path = f'/content/drive/MyDrive/{MODEL_NAME}_config.json'
with open(config_path, 'w') as f:
    json.dump(model_config_entry, f, indent=2)

print("✅ Model configuration generated for your codebase")
print(f"📁 Saved to: {config_path}")

### 9.2 Generate Final Summary

In [ ]:
# Generate comprehensive training summary
summary_report = f"""
# 🎯 Training Complete - {MODEL_NAME} (Nested Structure Handling)

## 📊 Performance Metrics
- **Test Accuracy**: {test_accuracy*100:.2f}%
- **Validation Accuracy**: {val_accuracy*100:.2f}%
- **Top-3 Accuracy**: {test_top3_acc*100:.2f}%
- **Training Samples**: {len(train_paths):,}
- **Validation Samples**: {len(val_paths):,}
- **Test Samples**: {len(test_paths):,}
- **Total Classes**: {len(CLASS_NAMES)}

## ⏱️ Training Statistics
- **Total Training Time**: {total_training_time/3600:.1f} hours
- **Phase 1 (Frozen)**: {phase1_duration/60:.1f} minutes
- **Phase 2 (Fine-tuning)**: {phase2_duration/60:.1f} minutes

## 💾 Model Specifications
- **Architecture**: MobileNetV2 + Custom Head
- **Input Shape**: 224x224x3
- **Parameters**: {best_model.count_params():,}
- **Model Size**: {total_size/1024/1024:.1f} MB
- **Quantization**: 16-bit

## 🗂️ Data Structure Handling
- **Structure**: Simplified flat structure with _day/_night suffixes
- **Day/Night Information**: Preserved in filename suffixes
- **Invalid Class**: Handled correctly (direct images)
- **Stratified Splitting**: ✅ Applied
- **Class Balancing**: ✅ Used class weights

## 📁 Generated Files in Google Drive
1. `{MODEL_NAME}_tfjs/` - TensorFlow.js model files
2. `{MODEL_NAME}_config.json` - Configuration for your app
3. `{MODEL_NAME}_classification_report.txt` - Performance details
4. `{MODEL_NAME}_confusion_matrix.png` - Visual analysis
5. `{MODEL_NAME}_training_summary.md` - This summary

## 🚀 Next Steps
1. Download model files from Google Drive
2. Upload to your hosting platform (Vercel, Netlify, AWS S3, etc.)
3. Update your models.json configuration
4. Test integration with your existing application
5. Monitor performance in production

## 🎉 Success!
Your custom moisture detection model successfully uses the simplified flat structure with preserved day/night information 
and is ready for deployment with {test_accuracy*100:.1f}% test accuracy!
"""

# Save summary
summary_path = f'/content/drive/MyDrive/{MODEL_NAME}_training_summary.md'
with open(summary_path, 'w') as f:
    f.write(summary_report)

print("📋 Training Summary:")
print("=" * 80)
print(summary_report)

print("\n🎊 Training completed successfully!")
print(f"⏰ Total time: {total_training_time/3600:.1f} hours")
print(f"🎯 Final test accuracy: {test_accuracy*100:.2f}%")
print("📱 Ready for integration with your application!")